# Real NEXRAD case: KLWX near Washington, DC

This notebook replaces the synthetic frames with two real NOAA NEXRAD Level II volume scans from **KLWX (Sterling, Virginia)**. The case is the June 5, 2024 severe-weather event near Poolesville, Maryland, west of Washington, DC.

The National Weather Service documented an EF-1 tornado beginning at 7:01 PM EDT (23:01 UTC). We use scans beginning at **23:04:37 UTC** and **23:10:39 UTC**, so this is a documented, reproducible storm-motion exercise—not a claim that this baseline detects a tornado.

## Provenance

- Radar: `KLWX`, Sterling, Virginia
- Source: [NOAA / Unidata public NEXRAD Level II archive](https://registry.opendata.aws/noaa-nexrad/)
- Files: `KLWX20240605_230437_V06`, `KLWX20240605_231039_V06`, `KLWX20240605_231643_V06`, and `KLWX20240605_232308_V06`
- Event context: [NWS Baltimore/Washington June 5, 2024 tornado event statement](https://www.weather.gov/lwx/June052024Tornado)

These source files are public. The download cell uses unsigned S3 access, so it does not need AWS credentials.

In [ ]:
# Run once in a fresh SageMaker kernel.
%pip install -e "../.[nexrad]"

In [ ]:
from pathlib import Path
import sys

import boto3
from botocore import UNSIGNED
from botocore.config import Config
import matplotlib.pyplot as plt
import numpy as np

repo = Path.cwd().parent
sys.path.insert(0, str(repo / 'src'))
from storm_motion.nexrad import load_lowest_reflectivity_sweep
from storm_motion.tracking import RadarFrame, detect_cells, match_cells

data_dir = repo / 'data' / 'klwx_20240605'
data_dir.mkdir(parents=True, exist_ok=True)
bucket = 'unidata-nexrad-level2'
keys = [
    '2024/06/05/KLWX/KLWX20240605_230437_V06',
    '2024/06/05/KLWX/KLWX20240605_231039_V06',
    '2024/06/05/KLWX/KLWX20240605_231643_V06',
    '2024/06/05/KLWX/KLWX20240605_232308_V06',
]
s3 = boto3.client('s3', region_name='us-east-1', config=Config(signature_version=UNSIGNED))
print(f'Repository: {repo}')

In [ ]:
# Download only if the four ~13 MB volume scans are not already present.
paths = []
for key in keys:
    path = data_dir / Path(key).name
    if not path.exists():
        print(f'Downloading s3://{bucket}/{key}')
        s3.download_file(bucket, key, str(path))
    else:
        print(f'Using cached file: {path.name}')
    paths.append(path)

frames = [load_lowest_reflectivity_sweep(path) for path in paths]
previous_frame, current_frame, *verification_frames = frames
for index, frame in enumerate(frames):
    print(f'Scan {index}: {frame.observed_at.isoformat()}')

In [ ]:
# Focus on a 70 km x 60 km window around Poolesville, MD, relative to KLWX.
# Coordinates are radar-relative: x is east/west and y is north/south.
roi_km = dict(xmin=-35, xmax=35, ymin=-10, ymax=50)

def in_roi(frame):
    x_km, y_km = frame.x_m / 1_000, frame.y_m / 1_000
    inside = (x_km >= roi_km['xmin']) & (x_km <= roi_km['xmax']) & (y_km >= roi_km['ymin']) & (y_km <= roi_km['ymax'])
    reflectivity = np.where(inside, frame.reflectivity_dbz, np.nan)
    return RadarFrame(frame.observed_at, reflectivity, frame.x_m, frame.y_m)

roi_frames = [in_roi(frame) for frame in frames]
previous_roi, current_roi, *verification_rois = roi_frames
fig, axes = plt.subplots(2, 2, figsize=(13, 12), sharex=True, sharey=True)
for ax, frame, label in zip(axes.flat, roi_frames, ['t0', 't+6 min', 't+12 min', 't+18 min']):
    image = ax.pcolormesh(frame.x_m / 1_000, frame.y_m / 1_000, frame.reflectivity_dbz, cmap='turbo', vmin=0, vmax=65)
    ax.set(title=f'{label}: {frame.observed_at:%Y-%m-%d %H:%M:%S UTC}', xlabel='km east of KLWX', ylabel='km north of KLWX', xlim=(roi_km['xmin'], roi_km['xmax']), ylim=(roi_km['ymin'], roi_km['ymax']), aspect='equal')
fig.colorbar(image, ax=axes.ravel().tolist(), label='reflectivity (dBZ)')
plt.show()

In [ ]:
# Detect and match high-reflectivity cells. Tune these two values as part of the experiment.
threshold_dbz, min_gates = 35.0, 30
previous_cells = detect_cells(previous_roi, threshold_dbz=threshold_dbz, min_gates=min_gates)
current_cells = detect_cells(current_roi, threshold_dbz=threshold_dbz, min_gates=min_gates)
elapsed_seconds = (current_roi.observed_at - previous_roi.observed_at).total_seconds()
matches = match_cells(previous_cells, current_cells, elapsed_seconds=elapsed_seconds, max_distance_m=30_000)
if not matches:
    # Cells can split or change shape enough that their centroids move beyond
    # the baseline's normal 30 km association gate between volume scans.
    print('No match within 30 km; retrying with a 60 km diagnostic association gate.')
    matches = match_cells(previous_cells, current_cells, elapsed_seconds=elapsed_seconds, max_distance_m=60_000)

print(f'Detected {len(previous_cells)} cells at t0 and {len(current_cells)} cells at t+6 min.')
for item in matches:
    print(f'{item.previous_cell_id} → {item.current_cell_id}: {item.speed_mps:.1f} m/s at {item.bearing_degrees:.0f}°')
if not matches:
    raise RuntimeError(f'No matches found from {len(previous_cells)} previous and {len(current_cells)} current cells. Try lowering threshold_dbz or min_gates.')

In [ ]:
# Plot the strongest matched cell and its 10/20/30 minute constant-velocity nowcast.
current_by_id = {cell.cell_id: cell for cell in current_cells}
selected = max(matches, key=lambda item: current_by_id[item.current_cell_id].max_reflectivity_dbz)
old_cell = next(cell for cell in previous_cells if cell.cell_id == selected.previous_cell_id)
new_cell = current_by_id[selected.current_cell_id]
dx_m, dy_m = new_cell.centroid_x_m - old_cell.centroid_x_m, new_cell.centroid_y_m - old_cell.centroid_y_m
forecast_minutes = np.array([10, 20, 30])
forecast_x_m = new_cell.centroid_x_m + selected.speed_mps * np.sin(np.deg2rad(selected.bearing_degrees)) * forecast_minutes * 60
forecast_y_m = new_cell.centroid_y_m + selected.speed_mps * np.cos(np.deg2rad(selected.bearing_degrees)) * forecast_minutes * 60

fig, ax = plt.subplots(figsize=(9, 7))
image = ax.pcolormesh(current_roi.x_m / 1_000, current_roi.y_m / 1_000, current_roi.reflectivity_dbz, cmap='turbo', vmin=0, vmax=65)
ax.arrow(old_cell.centroid_x_m / 1_000, old_cell.centroid_y_m / 1_000, dx_m / 1_000, dy_m / 1_000, width=.25, color='white', length_includes_head=True)
ax.plot([new_cell.centroid_x_m / 1_000, *list(forecast_x_m / 1_000)], [new_cell.centroid_y_m / 1_000, *list(forecast_y_m / 1_000)], '--', color='white', linewidth=2, label='projected path')
ax.scatter(new_cell.centroid_x_m / 1_000, new_cell.centroid_y_m / 1_000, s=110, c='white', edgecolors='black', zorder=3, label='current cell')
ax.scatter(forecast_x_m / 1_000, forecast_y_m / 1_000, s=100, c='#ffdf00', edgecolors='black', zorder=3, label='forecast center')
for minutes, x, y in zip(forecast_minutes, forecast_x_m, forecast_y_m):
    ax.annotate(f'+{minutes} min', (x / 1_000, y / 1_000), xytext=(6, 6), textcoords='offset points', color='white', weight='bold')
ax.set(title=f'KLWX storm-cell nowcast — {current_roi.observed_at:%Y-%m-%d %H:%M UTC}', xlabel='km east of KLWX', ylabel='km north of KLWX', xlim=(roi_km['xmin'], roi_km['xmax']), ylim=(roi_km['ymin'], roi_km['ymax']), aspect='equal')
ax.legend(loc='upper left'); fig.colorbar(image, ax=ax, label='reflectivity (dBZ)'); plt.show()
print(f'Selected matched cell: {selected.speed_mps:.1f} m/s, bearing {selected.bearing_degrees:.0f}°. This is a cell-motion estimate, not a tornado-track forecast.')

## Predicted versus observed

The following cell keeps the motion vector estimated from the first two scans fixed. For each later scan, it compares that forecast position with the detected cell whose centroid is closest to the forecast. The distance between the yellow forecast point and cyan observed point is the forecast-position error.

In [ ]:
# Validate the initial motion estimate against the two later scans.
velocity_x_mps, velocity_y_mps = dx_m / elapsed_seconds, dy_m / elapsed_seconds
evaluations = []

for frame in verification_rois:
    lead_seconds = (frame.observed_at - current_roi.observed_at).total_seconds()
    predicted_x_m = new_cell.centroid_x_m + velocity_x_mps * lead_seconds
    predicted_y_m = new_cell.centroid_y_m + velocity_y_mps * lead_seconds
    observed_cells = detect_cells(frame, threshold_dbz=threshold_dbz, min_gates=min_gates)
    if not observed_cells:
        print(f'No cells detected for {frame.observed_at:%H:%M UTC}; try lowering the threshold.')
        continue
    observed = min(observed_cells, key=lambda cell: np.hypot(cell.centroid_x_m - predicted_x_m, cell.centroid_y_m - predicted_y_m))
    error_m = np.hypot(observed.centroid_x_m - predicted_x_m, observed.centroid_y_m - predicted_y_m)
    evaluations.append((frame, lead_seconds, predicted_x_m, predicted_y_m, observed, error_m))

fig, axes = plt.subplots(1, len(evaluations), figsize=(8 * len(evaluations), 7), squeeze=False)
for ax, (frame, lead_seconds, predicted_x_m, predicted_y_m, observed, error_m) in zip(axes.flat, evaluations):
    image = ax.pcolormesh(frame.x_m / 1_000, frame.y_m / 1_000, frame.reflectivity_dbz, cmap='turbo', vmin=0, vmax=65)
    ax.scatter(predicted_x_m / 1_000, predicted_y_m / 1_000, s=130, c='#ffdf00', edgecolors='black', zorder=3, label='predicted center')
    ax.scatter(observed.centroid_x_m / 1_000, observed.centroid_y_m / 1_000, s=130, c='#00e5ff', edgecolors='black', zorder=3, label='observed center')
    ax.plot([predicted_x_m / 1_000, observed.centroid_x_m / 1_000], [predicted_y_m / 1_000, observed.centroid_y_m / 1_000], color='white', linewidth=2)
    ax.set(title=f'+{lead_seconds / 60:.0f} min: error {error_m / 1_000:.1f} km', xlabel='km east of KLWX', ylabel='km north of KLWX', xlim=(roi_km['xmin'], roi_km['xmax']), ylim=(roi_km['ymin'], roi_km['ymax']), aspect='equal')
    ax.legend(loc='upper left')
fig.colorbar(image, ax=axes.ravel().tolist(), label='reflectivity (dBZ)')
plt.show()

for frame, lead_seconds, _, _, _, error_m in evaluations:
    print(f'{frame.observed_at:%H:%M:%S UTC}  |  +{lead_seconds / 60:.1f} min  |  position error: {error_m / 1_000:.2f} km')